
# M115 — TP2 : Préparation et contrôle qualité des données

### Jeu de données : Online Retail

---

**Module :** Business Intelligence 
**Enseignante :** Pr. Amina Aboulmira  
**Environnement :** Kaggle Notebook  
**Langage :** Python  
**Bibliothèques :** Pandas, NumPy, Pathlib, hashlib, html  
**Réalisé par :** Meriem Nassih & Nisrine Qacem


---

## 1. Introduction

Dans le cadre du module **Business Intelligence (M115)**, ce travail pratique consiste à réaliser un processus complet de préparation et de contrôle qualité des données avec Python.

Le jeu de données utilisé est **Online Retail**, qui contient des informations sur les transactions commerciales d'une entreprise de vente en ligne.

Il comprend notamment :

- Les numéros de facture (`InvoiceNo`)
- Les codes produits (`StockCode`)
- Les descriptions (`Description`)
- Les quantités (`Quantity`)
- Les dates de transaction (`InvoiceDate`)
- Les prix unitaires (`UnitPrice`)
- Les identifiants clients (`CustomerID`)
- Les pays (`Country`)

## 2. Objectifs du TP

Les principaux objectifs sont :

1. Charger les données et assurer leur traçabilité.
2. Réaliser un audit initial de la qualité des données.
3. Identifier et isoler les doublons exacts.
4. Normaliser les champs et contrôler les types.
5. Classifier les lignes selon un référentiel commun.
6. Identifier les motifs d'exclusion.
7. Appliquer les filtres successifs et construire un journal.
8. Vérifier la cohérence et la conservation des lignes.
9. Exporter les données préparées et les fichiers d'audit.

## 3. Méthodologie

Le traitement est organisé en **neuf étapes successives**.

Chaque étape produit des résultats intermédiaires conservés dans des DataFrames distincts afin de garantir la traçabilité et la reproductibilité des traitements.

**Principe fondamental :** aucune ligne ne doit être supprimée sans justification. Les doublons et les lignes exclues sont conservés séparément pour permettre leur audit.




# Étape 01 — Chargement des données et traçabilité

## 1. Objectif

Cette première étape consiste à charger le fichier **Online Retail.xlsx** dans Kaggle et à vérifier la structure des données.

Elle permet également de conserver des informations sur la provenance des lignes.

## 2. Méthode utilisée

Les opérations effectuées sont :

1. Importer les bibliothèques Python nécessaires.
2. Rechercher le fichier Excel dans `/kaggle/input`.
3. Charger les données dans un DataFrame nommé `brut`.
4. Vérifier les huit colonnes métier attendues.
5. Ajouter trois colonnes de traçabilité :
   - `SourceIndex` : position de la ligne dans le DataFrame initial.
   - `SourceRow` : numéro de la ligne dans le fichier Excel.
   - `SourceFile` : nom du fichier source.
6. Calculer l'empreinte SHA-256 du fichier.

## 3. Outils utilisés

- **Pandas :** lecture et manipulation des données.
- **NumPy :** création des index de provenance.
- **Pathlib :** recherche et gestion des fichiers.
- **Hashlib :** calcul de l'empreinte SHA-256.




In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import html
import hashlib

colonnes_source = [
    "InvoiceNo", "StockCode", "Description", "Quantity",
    "InvoiceDate", "UnitPrice", "CustomerID", "Country"
]

fichiers = sorted(Path("/kaggle/input").rglob("Online Retail.xlsx"))

assert len(fichiers) == 1, "Ajouter un seul fichier Online Retail.xlsx dans les données Kaggle."

source = fichiers[0]
sortie = Path("/kaggle/working")
sortie.mkdir(parents=True, exist_ok=True)

brut = pd.read_excel(source)

assert list(brut.columns) == colonnes_source, "Vérifier le contrat des huit colonnes."

brut["SourceIndex"] = np.arange(len(brut))
brut["SourceRow"] = brut["SourceIndex"] + 2
brut["SourceFile"] = source.name

empreinte = hashlib.sha256(source.read_bytes()).hexdigest()

print("Empreinte du fichier :", empreinte)
print("Lignes :", len(brut))
print("Colonnes métier :", len(colonnes_source))

print(
    brut[colonnes_source + ["SourceIndex", "SourceRow"]]
    .head()
    .to_string(index=False)
)



## 4. Résultats obtenus

Après l'exécution du code, les résultats suivants ont été obtenus :

| Indicateur | Résultat |
|---|---|
| Fichier source | Online Retail.xlsx |
| Nombre de lignes | 541 909 |
| Nombre de colonnes métier | 8 |
| Première valeur SourceIndex | 0 |
| Première valeur SourceRow | 2 |

**Empreinte SHA-256 du fichier :**

`43465a06f2ccf7c8b5bd2892bc7defb52f97487934fe93b16ae4c3936424676d`

## 5. Interprétation

Le fichier a été chargé avec succès et contient **541 909 lignes et 8 colonnes métier**.

Les colonnes de provenance ont été ajoutées afin de faciliter le suivi des lignes pendant les différentes transformations.

L'empreinte SHA-256 permet d'identifier précisément le fichier source utilisé.

## 6. Conclusion

**Le chargement et la traçabilité des données sont validés.**

Le DataFrame `brut` constitue la référence initiale pour les étapes suivantes.




# Étape 02 — Audit initial de la qualité des données

## 1. Objectif

L'objectif de cette étape est d'analyser la qualité des données brutes avant toute opération de nettoyage.

Nous cherchons à identifier :

- Les types des colonnes.
- Les valeurs manquantes.
- Les lignes correspondant à des annulations.
- Les copies exactes supplémentaires.

**Important :** aucune ligne n'est supprimée pendant cette étape.

## 2. Méthodologie

Nous utilisons les fonctions suivantes :

- `dtype` : identifier le type des données.
- `isna().sum()` : compter les valeurs manquantes.
- `startswith("C")` : identifier les factures d'annulation.
- `duplicated()` : détecter les copies exactes sur les huit colonnes métier.

Les résultats sont enregistrés dans le DataFrame `audit_initial`.

## 3. Résultats obtenus



In [ ]:

audit_initial = pd.DataFrame({
    "Colonne": colonnes_source,
    "Type_observe": [str(brut[c].dtype) for c in colonnes_source],
    "Valeurs_manquantes": [int(brut[c].isna().sum()) for c in colonnes_source]
})

print(audit_initial.to_string(index=False))

print(
    "Commandes commençant par C :",
    brut["InvoiceNo"].astype(str).str.startswith("C").sum()
)

print(
    "Copies exactes supplémentaires :",
    brut.duplicated(subset=colonnes_source, keep="first").sum()
)


### 3.1. Types et valeurs manquantes

| Colonne | Type observé | Valeurs manquantes |
|---|---|---:|
| InvoiceNo | object | 0 |
| StockCode | object | 0 |
| Description | object | 1 454 |
| Quantity | int64 | 0 |
| InvoiceDate | datetime64[ns] | 0 |
| UnitPrice | float64 | 0 |
| CustomerID | float64 | 135 080 |
| Country | object | 0 |

### 3.2. Autres anomalies détectées

| Contrôle | Nombre |
|---|---:|
| Factures commençant par C | 9 288 |
| Copies exactes supplémentaires | 5 268 |

## 4. Analyse et interprétation

L'audit initial révèle plusieurs observations importantes.

### Valeurs manquantes

La colonne `CustomerID` contient **135 080 valeurs manquantes**.

Ces lignes ne doivent pas être supprimées automatiquement, car elles peuvent correspondre à des ventes exploitables.

La colonne `Description` contient également **1 454 valeurs manquantes**.

### Types des données

La colonne `InvoiceDate` est déjà reconnue comme une date (`datetime64[ns]`).

Les colonnes `Quantity` et `UnitPrice` possèdent des types numériques adaptés aux calculs.

### Annulations

**9 288 lignes** possèdent un numéro de facture commençant par `C`.
Ces lignes seront prises en compte dans les règles d'exclusion.

### Doublons

**5 268 copies exactes supplémentaires** ont été détectées.

Elles seront isolées à l'étape suivante.

## 5. Conclusion

L'audit a permis d'identifier les principales anomalies sans modifier les données.

Les résultats serviront de référence pour les opérations de préparation et de contrôle qualité.

# Étape 03 — Dédoublonnage exact et conservation des copies

## 1. Objectif

Cette étape consiste à identifier et isoler les copies exactes supplémentaires présentes dans le fichier.

Le dédoublonnage doit être effectué **avant la normalisation**, en comparant les huit colonnes métier originales.

L'objectif est de supprimer les répétitions du périmètre de traitement tout en conservant les copies dans un fichier d'audit.

## 2. Méthodologie

La méthode `duplicated()` est utilisée avec les paramètres :

- `subset=colonnes_source` : comparaison sur les huit colonnes métier.
- `keep="first"` : conservation de la première occurrence.

Deux DataFrames sont créés :

- `doublons_exacts` : copies exactes supplémentaires.
- `staging_unique` : premières occurrences conservées.

Des assertions permettent de vérifier que toutes les lignes sont prises en compte et qu'aucun doublon exact ne subsiste dans `staging_unique`.

In [ ]:

masque_doublon = brut.duplicated(
    subset=colonnes_source,
    keep="first"
)

doublons_exacts = brut.loc[masque_doublon].copy()

staging_unique = brut.loc[~masque_doublon].copy()

assert len(brut) == len(doublons_exacts) + len(staging_unique)

assert not staging_unique.duplicated(subset=colonnes_source).any()

print("Copies retirées :", len(doublons_exacts))
print("Lignes uniques :", len(staging_unique))

print(
    doublons_exacts[
        ["SourceIndex", "InvoiceNo", "StockCode", "Quantity"]
    ].head().to_string(index=False)
)


## 3. Résultats obtenus

| Indicateur | Nombre |
|---|---:|
| Lignes initiales | 541 909 |
| Copies exactes retirées | 5 268 |
| Lignes uniques conservées | 536 641 |

### Vérification de la conservation

**541 909 = 5 268 + 536 641**

Cette égalité confirme que les lignes initiales sont réparties entre les copies supplémentaires et les lignes conservées.

## 4. Interprétation

Le dédoublonnage a permis d'isoler **5 268 copies exactes supplémentaires**.

Les **536 641 lignes restantes** constituent le staging dédoublonné.

Les copies retirées restent disponibles dans `doublons_exacts`, avec leurs informations de provenance.

Cette méthode permet de garantir la traçabilité du nettoyage.

## 5. Conclusion

Le dédoublonnage exact est effectué avec succès.

Aucune ligne n'a été perdue sans justification.

# Étape 04 — Normalisation des données et contrôle du typage

## 1. Objectif

Cette étape vise à harmoniser les données textuelles et à vérifier les types des colonnes utilisées dans les calculs.

Les colonnes originales doivent rester inchangées afin de préserver les données sources.

## 2. Méthodologie

Un nouveau DataFrame `staging_type` est créé à partir de `staging_unique`.

### 2.1. Normalisation textuelle

Les colonnes suivantes sont normalisées :

- `StockCode`
- `Description`
- `Country`

Les opérations utilisées sont :

- `.astype("string")` : conversion en chaîne de caractères.
- `.str.strip()` : suppression des espaces inutiles.
- `.str.upper()` : conversion en majuscules.

Les résultats sont enregistrés dans des colonnes portant le suffixe `_norm`.

### 2.2. Normalisation des identifiants

Deux nouvelles colonnes sont créées :

- `InvoiceNo_norm`
- `CustomerID_norm`

Les valeurs manquantes de `CustomerID` sont préservées.

### 2.3. Conversion numérique

Les colonnes `Quantity` et `UnitPrice` sont converties avec `pd.to_numeric()`.

Les nouvelles colonnes sont :

- `Quantity_num`
- `UnitPrice_num`

Des indicateurs `_echec` permettent de détecter les erreurs de conversion.

### 2.4. Conversion des dates

La colonne `InvoiceDate_dt` est créée avec `pd.to_datetime()`.

Un indicateur `InvoiceDate_echec` permet de détecter les dates non convertibles.

In [ ]:

staging_type = staging_unique.copy()

# Normalisation des colonnes textuelles
for c in ["StockCode", "Description", "Country"]:
    staging_type[c + "_norm"] = (
        staging_type[c]
        .astype("string")
        .str.strip()
        .str.upper()
    )

# Normalisation du numéro de facture
staging_type["InvoiceNo_norm"] = (
    staging_type["InvoiceNo"]
    .astype("string")
    .str.strip()
)

# Normalisation de l'identifiant client
staging_type["CustomerID_norm"] = (
    staging_type["CustomerID"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

# Conversion de la quantité et du prix
for c in ["Quantity", "UnitPrice"]:
    staging_type[c + "_num"] = pd.to_numeric(
        staging_type[c],
        errors="coerce"
    )

    staging_type[c + "_echec"] = (
        staging_type[c].notna()
        & staging_type[c + "_num"].isna()
    )

# Conversion des dates
if pd.api.types.is_datetime64_any_dtype(staging_type["InvoiceDate"]):
    staging_type["InvoiceDate_dt"] = pd.to_datetime(
        staging_type["InvoiceDate"],
        errors="coerce"
    )
else:
    staging_type["InvoiceDate_dt"] = pd.to_datetime(
        staging_type["InvoiceDate"],
        format="%d/%m/%Y %H:%M",
        errors="coerce"
    )

staging_type["InvoiceDate_echec"] = (
    staging_type["InvoiceDate"].notna()
    & staging_type["InvoiceDate_dt"].isna()
)


In [ ]:

anomalies_typage = staging_type.loc[
    staging_type[
        ["Quantity_echec", "UnitPrice_echec", "InvoiceDate_echec"]
    ].any(axis=1)
    | staging_type["Quantity_num"].isna()
    | staging_type["UnitPrice_num"].isna()
    | staging_type["InvoiceDate_dt"].isna()
].copy()

print(
    staging_type[
        ["StockCode", "StockCode_norm", "Quantity", "Quantity_num"]
    ].head().to_string(index=False)
)

print("Anomalies à diagnostiquer :", len(anomalies_typage))

assert len(staging_type) == len(staging_unique)

assert anomalies_typage.empty, (
    "Ne pas inventer un filtre : montrer les anomalies à l'enseignante."
)


## 3. Résultats obtenus

| Contrôle | Résultat |
|---|---:|
| Lignes avant normalisation | 536 641 |
| Lignes après normalisation | 536 641 |
| Anomalies de typage | 0 |

## 4. Interprétation

La normalisation a été réalisée sans supprimer de lignes.

Les champs textuels disposent désormais de versions homogènes.

Les quantités, les prix et les dates ont été contrôlés.

**Aucune anomalie de typage n'a été détectée.**

Les valeurs originales restent disponibles pour les besoins de traçabilité.

## 5. Conclusion

Les données sont normalisées et les conversions nécessaires ont été validées.

Le DataFrame `staging_type` est prêt pour la classification métier.

# Étape 05 — Classification des lignes selon le référentiel commun

## 1. Objectif

Cette étape consiste à identifier la nature métier de chaque ligne.

L'objectif est de distinguer les produits commerciaux des frais, ajustements, remises, commissions et autres opérations.

La classification repose sur le référentiel commun défini dans le protocole du TP.

## 2. Méthodologie

Un DataFrame `integration` est créé à partir de `staging_type`.

### 2.1. Classification par code

Un dictionnaire `codes_non_commerciaux` associe certains codes produits à des catégories spécifiques.

Exemples :

| Code | Catégorie |
|---|---|
| POST | FRAIS_PORT |
| M | AJUSTEMENT |
| D | REMISE |
| AMAZONFEE | COMMISSION |
| S | ECHANTILLON |

Les codes absents du dictionnaire sont classés par défaut comme `PRODUIT_COMMERCIAL`.

### 2.2. Détection des descriptions suspectes

Une expression régulière recherche des termes tels que :

- ADJUSTMENT
- MANUAL
- POSTAGE
- DISCOUNT
- COMMISSION

Les lignes classées par défaut comme produits commerciaux et possédant une description suspecte sont reclassées en `A_VERIFIER`.

### 2.3. Traçabilité de la classification

Deux colonnes sont créées :

- `TypeLigne` : catégorie métier.
- `OrigineClassification` : origine de la règle appliquée.

In [ ]:

codes_non_commerciaux = {
    "POST": "FRAIS_PORT",
    "DOT": "FRAIS_PORT",
    "C2": "FRAIS_PORT",
    "M": "AJUSTEMENT",
    "B": "AJUSTEMENT",
    "PADS": "AJUSTEMENT",
    "D": "REMISE",
    "BANK CHARGES": "FRAIS_BANCAIRES",
    "AMAZONFEE": "COMMISSION",
    "CRUK": "COMMISSION",
    "S": "ECHANTILLON",
    "GIFT_0001_10": "BON_CADEAU",
    "GIFT_0001_20": "BON_CADEAU",
    "GIFT_0001_30": "BON_CADEAU",
    "GIFT_0001_40": "BON_CADEAU",
    "GIFT_0001_50": "BON_CADEAU"
}

integration = staging_type.copy()

code = integration["StockCode_norm"].fillna("")
description = integration["Description_norm"].fillna("")

integration["TypeLigne"] = (
    code.map(codes_non_commerciaux)
    .fillna("PRODUIT_COMMERCIAL")
)

integration["OrigineClassification"] = np.where(
    code.isin(codes_non_commerciaux),
    "REFERENTIEL_CODE",
    "REGLE_PAR_DEFAUT"
)


In [ ]:

motif = (
    r"\b(?:ADJUSTMENT|MANUAL|POSTAGE|BANK CHARGES|AMAZON FEE|"
    r"DISCOUNT|SAMPLES?|COMMISSION|CARRIAGE|DOTCOM POSTAGE)\b"
)

suspect = description.str.contains(
    motif,
    regex=True,
    na=False
)

a_verifier = (
    suspect
    & integration["TypeLigne"].eq("PRODUIT_COMMERCIAL")
)

integration.loc[a_verifier, "TypeLigne"] = "A_VERIFIER"

integration.loc[
    a_verifier,
    "OrigineClassification"
] = "DESCRIPTION_SUSPECTE"


In [ ]:

classification_effectifs = (
    integration["TypeLigne"]
    .value_counts()
    .rename_axis("TypeLigne")
    .reset_index(name="Nombre")
)

assert len(integration) == len(staging_type)

assert classification_effectifs["Nombre"].sum() == len(integration)

print(classification_effectifs.to_string(index=False))

print(
    integration[
        [
            "SourceIndex",
            "StockCode",
            "Description",
            "TypeLigne",
            "OrigineClassification"
        ]
    ].head().to_string(index=False)
)


## 3. Résultats obtenus

| TypeLigne | Nombre |
|---|---:|
| PRODUIT_COMMERCIAL | 533 490 |
| FRAIS_PORT | 2 110 |
| AJUSTEMENT | 574 |
| A_VERIFIER | 207 |
| REMISE | 77 |
| ECHANTILLON | 62 |
| COMMISSION | 50 |
| FRAIS_BANCAIRES | 37 |
| BON_CADEAU | 34 |
| **Total** | **536 641** |

## 4. Analyse et interprétation

La majorité des lignes ont été classées comme `PRODUIT_COMMERCIAL`.

Les autres catégories correspondent à des opérations qui ne doivent pas être automatiquement considérées comme des ventes de produits.

**207 lignes** ont été classées `A_VERIFIER` en raison de descriptions suspectes.

La somme des effectifs est égale à **536 641**, ce qui confirme que toutes les lignes du staging dédoublonné ont été classifiées.

## 5. Conclusion

La classification métier a été réalisée avec succès.

Chaque ligne possède désormais une catégorie et une origine de classification.

# Étape 06 — Identification et justification des motifs d'exclusion

## 1. Objectif

Cette étape consiste à déterminer les lignes admissibles au périmètre des ventes préparées.

Chaque exclusion doit être justifiée par une règle métier explicite.

Une même ligne peut présenter plusieurs motifs d'exclusion.

## 2. Règles appliquées

Quatre motifs sont définis :

| Motif | Condition |
|---|---|
| ANNULATION | InvoiceNo commence par C |
| QUANTITE_NON_POSITIVE | Quantity ≤ 0 |
| PRIX_NON_POSITIF | UnitPrice ≤ 0 |
| TYPE_HORS_VENTES | TypeLigne différent de PRODUIT_COMMERCIAL |

## 3. Méthodologie

Un DataFrame `flags` est créé pour représenter les quatre conditions sous forme de valeurs booléennes.

Deux colonnes sont ajoutées à `integration` :

### MotifsExclusion

Cette colonne contient les motifs justifiant l'exclusion d'une ligne.

Exemple :

`ANNULATION;QUANTITE_NON_POSITIVE`

### InclureFactVente

Cette colonne indique si la ligne est admissible :

- `True` : aucun motif d'exclusion.
- `False` : au moins un motif d'exclusion.

In [ ]:

integration["Annulation"] = (
    integration["InvoiceNo_norm"]
    .str.startswith("C", na=False)
)

flags = pd.DataFrame({
    "ANNULATION": integration["Annulation"],
    "QUANTITE_NON_POSITIVE": integration["Quantity_num"].le(0),
    "PRIX_NON_POSITIF": integration["UnitPrice_num"].le(0),
    "TYPE_HORS_VENTES": integration["TypeLigne"].ne("PRODUIT_COMMERCIAL")
}, index=integration.index)

# Pour chaque ligne, recueillir les noms des indicateurs vrais.
integration["MotifsExclusion"] = ""

for nom in flags.columns:
    positions = flags[nom]
    integration.loc[positions, "MotifsExclusion"] += nom + ";"

integration["MotifsExclusion"] = (
    integration["MotifsExclusion"].str.rstrip(";")
)

integration["InclureFactVente"] = ~flags.any(axis=1)

print(
    integration[
        ["SourceIndex", "TypeLigne",
         "InclureFactVente", "MotifsExclusion"]
    ].head(12).to_string(index=False)
)

print("Effectifs par motif (chevauchements possibles) :")
print(flags.sum().to_string())


## 4. Résultats obtenus

| Motif d'exclusion | Nombre |
|---|---:|
| ANNULATION | 9 251 |
| QUANTITE_NON_POSITIVE | 10 587 |
| PRIX_NON_POSITIF | 2 512 |
| TYPE_HORS_VENTES | 3 151 |

## 5. Analyse et interprétation

Les annulations et les quantités non positives représentent une part importante des anomalies identifiées.

Le nombre d'annulations est passé de **9 288** à **9 251** après le dédoublonnage.

Cette différence correspond à **37 copies exactes supplémentaires** concernant des lignes d'annulation.

Les effectifs des motifs peuvent se chevaucher. Il est donc incorrect de les additionner pour calculer le nombre total de lignes exclues.

Aucune suppression physique n'est réalisée à cette étape : les motifs sont simplement enregistrés.

## 6. Conclusion

Les règles d'exclusion ont été appliquées et documentées.

Chaque ligne dispose désormais d'une décision d'éligibilité et de motifs explicites.

# Étape 07 — Filtrage successif et journal de contrôle

## 1. Objectif

Cette étape consiste à appliquer les règles d'exclusion de manière successive.

L'objectif est de mesurer l'impact de chaque filtre sur :

- Le nombre de lignes.
- Les quantités.
- Les montants.
- Les identifiants clients manquants.

## 2. Méthodologie

Cinq étapes intermédiaires sont créées :

| DataFrame | Règle |
|---|---|
| etape_0 | Staging dédoublonné |
| etape_1 | Exclusion des annulations |
| etape_2 | Conservation des quantités positives |
| etape_3 | Conservation des prix positifs |
| etape_4 | Conservation des produits commerciaux |

Le DataFrame `journal` enregistre les indicateurs calculés à chaque étape.

Deux autres DataFrames sont créés :

- `preparee` : lignes admissibles.
- `audit_exclusions` : lignes exclues avec leurs motifs.

Une nouvelle colonne `MontantLigne` est calculée :

**MontantLigne = Quantity_num × UnitPrice_num**

In [ ]:

etape_0 = integration.copy()

etape_1 = etape_0.loc[
    ~etape_0["Annulation"]
].copy()

etape_2 = etape_1.loc[
    etape_1["Quantity_num"].gt(0)
].copy()

etape_3 = etape_2.loc[
    etape_2["UnitPrice_num"].gt(0)
].copy()

etape_4 = etape_3.loc[
    etape_3["TypeLigne"].eq("PRODUIT_COMMERCIAL")
].copy()


In [ ]:

def montant(df):
    return (df["Quantity_num"] * df["UnitPrice_num"]).sum()

etapes = [etape_0, etape_1, etape_2, etape_3, etape_4]

conditions = [
    "Staging dédoublonné",
    "InvoiceNo ne commence pas par C",
    "Quantity > 0",
    "UnitPrice > 0",
    "TypeLigne = PRODUIT_COMMERCIAL"
]

journal = pd.DataFrame([
    {
        "Etape": i,
        "Condition_conservee": conditions[i],
        "Retirees": None if i == 0 else len(etapes[i-1]) - len(df),
        "Restantes": len(df),
        "Quantite": df["Quantity_num"].sum(),
        "Montant": montant(df),
        "Clients_absents": int(df["CustomerID_norm"].isna().sum())
    }
    for i, df in enumerate(etapes)
])

print(
    journal.to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)


In [ ]:

preparee = etape_4.copy()

preparee["MontantLigne"] = (
    preparee["Quantity_num"] * preparee["UnitPrice_num"]
)

audit_exclusions = integration.loc[
    ~integration["InclureFactVente"]
].copy()

assert set(preparee["SourceIndex"]) == set(
    integration.loc[
        integration["InclureFactVente"], "SourceIndex"
    ]
)

print("Lignes préparées :", len(preparee))
print("Lignes exclues :", len(audit_exclusions))
print("Montant total préparé :", preparee["MontantLigne"].sum())


## 3. Résultats du filtrage

| Étape | Lignes retirées | Lignes restantes |
|---|---:|---:|
| 0 — Staging dédoublonné | — | 536 641 |
| 1 — Annulations | 9 251 | 527 390 |
| 2 — Quantité positive | 1 336 | 526 054 |
| 3 — Prix positif | 1 176 | 524 878 |
| 4 — Produit commercial | 2 514 | 522 364 |

## 4. Bilan final

| Indicateur | Valeur |
|---|---:|
| Ventes préparées | 522 364 |
| Lignes exclues | 14 277 |
| Quantité totale conservée | 5 561 117 |
| Montant total préparé | 10 244 661,18 |
| Ventes sans CustomerID conservées | 131 355 |

## 5. Analyse et interprétation

Le filtrage a permis de passer de **536 641 lignes dédoublonnées** à **522 364 lignes admissibles**.

Les filtres sont appliqués dans un ordre précis. Le nombre de lignes retirées à une étape dépend donc des filtres précédents.

Le montant peut augmenter après la suppression de certaines lignes, notamment lorsque celles-ci possèdent un montant négatif.

Les ventes admissibles sans identifiant client sont conservées.

Le montant obtenu correspond au périmètre des ventes préparées et ne représente pas nécessairement le chiffre d'affaires comptable net.

## 6. Conclusion

Les filtres successifs ont été appliqués avec succès.

Le journal permet de suivre et de justifier les transformations réalisées.

# Étape 08 — Contrôles de cohérence et de provenance

## 1. Objectif

Cette étape consiste à vérifier l'intégrité des données préparées.

Les contrôles doivent garantir que les ventes admissibles sont conservées, que les exclusions sont correctement identifiées et que les montants calculés sont exacts.

## 2. Méthodologie

Plusieurs contrôles sont réalisés à l'aide d'assertions Python.

### 2.1. Unicité des identifiants

Chaque ligne préparée doit posséder un `SourceIndex` unique.

### 2.2. Conservation des ventes admissibles

Les lignes présentes dans `preparee` doivent correspondre exactement aux lignes répondant aux règles d'éligibilité.

### 2.3. Conservation des ventes sans CustomerID

Les ventes admissibles ne doivent pas être supprimées uniquement parce que l'identifiant client est absent.

### 2.4. Séparation des ventes et des exclusions

Une même ligne ne doit pas apparaître simultanément dans `preparee` et `audit_exclusions`.

### 2.5. Conservation des lignes d'origine

Toutes les lignes initiales doivent être retrouvées dans les trois ensembles :

- `doublons_exacts`
- `audit_exclusions`
- `preparee`

### 2.6. Vérification des montants

Les montants sont contrôlés avec `np.allclose()` et `np.isclose()`.

Chaque montant doit respecter la formule :

**MontantLigne = Quantity_num × UnitPrice_num**

In [ ]:

attendues = integration.loc[
    (~integration["Annulation"])
    & (integration["Quantity_num"] > 0)
    & (integration["UnitPrice_num"] > 0)
    & integration["TypeLigne"].eq("PRODUIT_COMMERCIAL")
]

ids_attendus = set(attendues["SourceIndex"])

ids_observes = set(preparee["SourceIndex"])

ids_clients_absents = set(
    attendues.loc[
        attendues["CustomerID_norm"].isna(),
        "SourceIndex"
    ]
)

assert preparee["SourceIndex"].is_unique

assert ids_observes == ids_attendus, (
    "Ventes perdues ou ajoutées."
)

assert ids_clients_absents <= ids_observes, (
    "Vente au client absent perdue."
)

assert set(preparee["SourceIndex"]).isdisjoint(
    set(audit_exclusions["SourceIndex"])
)

assert set(integration["SourceIndex"]) == (
    ids_observes | set(audit_exclusions["SourceIndex"])
)

assert len(brut) == (
    len(doublons_exacts)
    + len(audit_exclusions)
    + len(preparee)
)


In [ ]:

controle_lignes = attendues[
    ["SourceIndex", "Quantity_num", "UnitPrice_num"]
].merge(
    preparee[["SourceIndex", "MontantLigne"]],
    on="SourceIndex",
    how="outer",
    validate="one_to_one",
    indicator=True
)

assert controle_lignes["_merge"].eq("both").all()

assert np.allclose(
    controle_lignes["Quantity_num"]
    * controle_lignes["UnitPrice_num"],
    controle_lignes["MontantLigne"],
    rtol=1e-12,
    atol=1e-9
)

assert np.isclose(
    montant(attendues),
    preparee["MontantLigne"].sum(),
    rtol=1e-12,
    atol=1e-9
)


In [ ]:

controle_final = pd.DataFrame({
    "Controle": [
        "Ventes préparées",
        "Lignes écartées après dédoublonnage",
        "Quantité totale conservée",
        "Somme MontantLigne",
        "Ventes conservées avec CustomerID manquant"
    ],
    "Valeur_observee": [
        len(preparee),
        len(audit_exclusions),
        preparee["Quantity_num"].sum(),
        preparee["MontantLigne"].sum(),
        preparee["CustomerID_norm"].isna().sum()
    ]
})

print(
    controle_final.to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)

print("Provenances et montants : contrôles exécutés sans échec.")


## 3. Résultats obtenus

| Contrôle | Valeur |
|---|---:|
| Ventes préparées | 522 364 |
| Lignes exclues | 14 277 |
| Quantité totale conservée | 5 561 117 |
| Somme MontantLigne | 10 244 661,18 |
| Ventes sans CustomerID conservées | 131 355 |

## 4. Vérification de la conservation

**541 909 = 5 268 + 14 277 + 522 364**

Cette égalité confirme que toutes les lignes du fichier initial sont prises en compte.

## 5. Analyse et interprétation

Toutes les assertions ont été exécutées sans erreur.

Les contrôles confirment :

- L'unicité des identifiants de provenance.
- La conservation des ventes admissibles.
- La séparation entre ventes et exclusions.
- La cohérence des montants calculés.
- La conservation des ventes sans identifiant client.

Le message suivant confirme le succès des vérifications :

`Provenances et montants : contrôles exécutés sans échec.`

## 6. Conclusion

Les contrôles de cohérence et de provenance sont validés.

Les données préparées respectent les règles définies dans le protocole.

# Étape 09 — Exportation des données et production des livrables

## 1. Objectif

Cette dernière étape consiste à exporter les résultats du traitement dans des fichiers CSV et à générer un rapport qualité au format HTML.

Les fichiers sont enregistrés dans le répertoire `/kaggle/working`.

## 2. Méthodologie

Les DataFrames sont exportés à l'aide de la méthode `to_csv()`.

Les paramètres utilisés sont :

- `index=False` : éviter l'ajout d'une colonne d'index.
- `encoding="utf-8-sig"` : faciliter la lecture des fichiers dans Excel.

Le rapport HTML est généré à partir des tableaux de diagnostic et de contrôle à l'aide de la méthode `to_html()`.

## 3. Fichiers générés

| Fichier | Contenu |
|---|---|
| TP2_L2_SOURCE_PREPAREE.csv | Ventes préparées |
| TP2_L2_STAGING_TRACE.csv | Données enrichies et traçabilité |
| TP2_L2_AUDIT_EXCLUSIONS.csv | Lignes exclues |
| TP2_L2_DOUBLONS_EXACTS.csv | Copies exactes supplémentaires |
| TP2_L2_JOURNAL_CONTROLES.csv | Journal des étapes |
| TP2_CORRECTION_IDEALE_M115.html | Rapport qualité |

In [ ]:

preparee.to_csv(
    sortie / "TP2_L2_SOURCE_PREPAREE.csv",
    index=False,
    encoding="utf-8-sig"
)

integration.to_csv(
    sortie / "TP2_L2_STAGING_TRACE.csv",
    index=False,
    encoding="utf-8-sig"
)

audit_exclusions.to_csv(
    sortie / "TP2_L2_AUDIT_EXCLUSIONS.csv",
    index=False,
    encoding="utf-8-sig"
)

doublons_exacts.to_csv(
    sortie / "TP2_L2_DOUBLONS_EXACTS.csv",
    index=False,
    encoding="utf-8-sig"
)

journal.to_csv(
    sortie / "TP2_L2_JOURNAL_CONTROLES.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Les cinq fichiers CSV ont été générés.")


In [ ]:

rapport = """<!doctype html>
<html lang="fr">
<meta charset="utf-8">
<title>M115 — Rapport qualité L2</title>
<style>
body {
    font-family: Arial;
    max-width: 1100px;
    margin: 30px auto;
    line-height: 1.5;
}
table { border-collapse: collapse; }
td, th {
    border: 1px solid #ccc;
    padding: 8px;
}
textarea {
    width: 95%;
    min-height: 80px;
}
@media print {
    textarea { border: 1px solid #999; }
}
</style>
<h1>M115 — TP2 · Rapport qualité L2</h1>
<p>Pr. AMINA ABOULMIRA · Étudiants : ____________________</p>
"""

rapport += (
    "<p>Fichier : "
    + html.escape(source.name)
    + "<br>SHA-256 : "
    + empreinte
    + "</p>"
)

tableaux = [
    ("Audit initial", audit_initial),
    ("Classification", classification_effectifs),
    ("Journal des étapes", journal),
    ("Contrôle final", controle_final)
]

for titre, tableau in tableaux:
    rapport += "<h2>" + titre + "</h2>"
    rapport += tableau.to_html(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
    rapport += (
        "<p>Interprétation et anomalie éventuelle :</p>"
        "<textarea></textarea>"
    )

rapport += """
<h2>Règles et preuves</h2>
<p>Joindre le script, commenter les copies exactes,
le client absent et les motifs multiples.
Le montant décrit le périmètre MVP,
pas un CA comptable net.</p>
<p>Les graphiques sont facultatifs :
les tableaux exacts restent prioritaires.</p>
<p>Pour conserver vos commentaires :
imprimer ce rapport en PDF après saisie.</p>
</html>
"""

(sortie / "TP2_CORRECTION_IDEALE_M115.html").write_text(
    rapport,
    encoding="utf-8"
)

print("Rapport HTML généré avec succès.")


In [ ]:

for nom in sorted(sortie.glob("TP2_*.csv")):
    print(nom.name)

print("TP2_CORRECTION_IDEALE_M115.html")
print(
    "À joindre : TP2_L2_SCRIPT_REPRODUCTIBLE.py "
    "téléchargé depuis la fiche, "
    "et votre notebook exécuté."
)


## 4. Résultats

Les cinq fichiers CSV et le rapport HTML ont été générés avec succès.

Ces fichiers permettent :

- De réutiliser les données préparées.
- De consulter les lignes exclues.
- De retrouver les copies exactes.
- De vérifier les règles appliquées.
- De documenter les contrôles qualité.

Le notebook exécuté et le script Python reproductible complètent les livrables.

## 5. Conclusion

Les données préparées et les fichiers d'audit sont disponibles dans Kaggle.

Le processus de préparation et de contrôle qualité est terminé.